In [104]:
#import pyspark
import findspark

In [2]:
!pip install findspark

In [3]:
findspark.find()

'D:\\Software\\spark-4.0.0-bin-hadoop3\\spark-4.0.0-bin-hadoop3'

In [4]:
#initiate spark
import pyspark
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
c = pyspark.SparkConf().setAppName("test_app").setMaster("local")
sc = pyspark.SparkContext(conf = c)
spark = SparkSession(sc)

In [5]:
from pyspark.sql.functions import *

In [7]:
data = spark.read.csv("D:\\PySpark\\Data Set\\Sample - Superstore v1.csv", header = True , inferSchema=True, escape='"')

In [8]:
data.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country/Region: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [9]:
data.count(),len(data.columns)

(9994, 21)

In [11]:
#Changing datatype-------

#data = data.withColumn("Sales", col("Sales").cast("int"))
#data = data.withColumn("Quantity", col("Quantity").cast("int"))
#data = data.withColumn("Discount", col("Discount").cast("int"))

In [13]:
data.show(3)

+------+--------------+----------+----------+--------------+-----------+-------------+-----------+--------------+------------+----------+-----------+-------+---------------+----------+------------+--------------------+--------+--------+--------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|Customer Name|    Segment|Country/Region|        City|     State|Postal Code| Region|     Product ID|  Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|    Profit|
+------+--------------+----------+----------+--------------+-----------+-------------+-----------+--------------+------------+----------+-----------+-------+---------------+----------+------------+--------------------+--------+--------+--------+----------+
|  2698|CA-2016-145317|18-03-2016|23-03-2016|Standard Class|   SM-20320|  Sean Miller|Home Office| United States|Jacksonville|   Florida|      32216|  South|TEC-MA-10002412|Technology|    Machines|Cisco TelePresenc...|22638.48|  

In [69]:
data1 = data.select("Segment", "Region","Category","Sales","Profit", "Quantity","Sub-Category")

In [70]:
data1.show(5)

+-----------+-------+----------+---------+----------+--------+------------+
|    Segment| Region|  Category|    Sales|    Profit|Quantity|Sub-Category|
+-----------+-------+----------+---------+----------+--------+------------+
|Home Office|  South|Technology| 22638.48|-1811.0784|       6|    Machines|
|  Corporate|Central|Technology| 17499.95|  8399.976|       5|     Copiers|
|   Consumer|   West|Technology| 13999.96| 6719.9808|       4|     Copiers|
|Home Office|   East|Technology|11199.968| 3919.9888|       4|     Copiers|
|   Consumer|   East|Technology| 10499.97| 5039.9856|       3|     Copiers|
+-----------+-------+----------+---------+----------+--------+------------+
only showing top 5 rows


In [71]:
# one Col one Aggregate
data1.groupby("Segment").sum("Sales").show(10)


+-----------+-----------------+
|    Segment|       sum(Sales)|
+-----------+-----------------+
|   Consumer|1161401.344999977|
|Home Office|429653.1485000006|
|  Corporate|706146.3667999947|
+-----------+-----------------+



In [72]:
data1.filter(col("Row ID")==6076).show(10,False)

+---------+------+---------------+------+------+--------+------------+
|Segment  |Region|Category       |Sales |Profit|Quantity|Sub-Category|
+---------+------+---------------+------+------+--------+------------+
|Corporate|West  |Office Supplies|270.62|2.7062|2       |Storage     |
+---------+------+---------------+------+------+--------+------------+



In [73]:
# 2 Columns one aggregatoin

In [74]:
data1.groupby("Segment","Region").agg(sum("Sales")).show(6)

+-----------+-------+------------------+
|    Segment| Region|        sum(Sales)|
+-----------+-------+------------------+
|Home Office|   West|136721.77699999965|
|  Corporate|   West|225855.27449999994|
|   Consumer|   West|362880.77300000144|
|   Consumer|   East|350908.16700000095|
|Home Office|Central| 91212.64399999991|
|  Corporate|Central| 157995.8127999998|
+-----------+-------+------------------+
only showing top 6 rows


In [75]:
# One Col 2 Aggregation

In [76]:
data1.groupby("Region").agg(sum("Sales"),sum("Profit")).show(6)

+-------+------------------+------------------+
| Region|        sum(Sales)|       sum(Profit)|
+-------+------------------+------------------+
|  South|391721.90500000044| 46749.43030000002|
|Central|501239.89080000174|39706.362499999916|
|   East| 678781.2399999928| 91522.78000000009|
|   West|  725457.824499995|108418.44890000018|
+-------+------------------+------------------+



In [77]:
# 2 Col 2 Aggregation

In [78]:
data1.groupby("Region","Segment").agg(sum("Sales"),sum("Profit")).show(6)

+-------+-----------+------------------+------------------+
| Region|    Segment|        sum(Sales)|       sum(Profit)|
+-------+-----------+------------------+------------------+
|Central|Home Office| 91212.64399999991|12438.412399999997|
|   West|Home Office|136721.77699999965|16530.414999999957|
|  South|Home Office|        74255.0015|4620.6343000000015|
|   East|Home Office| 127463.7259999999| 26709.21680000001|
|  South|   Consumer|195580.97100000046|26913.572799999987|
|  South|  Corporate| 121885.9325000001|        15215.2232|
+-------+-----------+------------------+------------------+
only showing top 6 rows


In [79]:
data1.groupby("Region","Segment").agg(sum("Sales"),sum("Profit"),avg("Profit")).show(6)

+-------+-----------+------------------+------------------+------------------+
| Region|    Segment|        sum(Sales)|       sum(Profit)|       avg(Profit)|
+-------+-----------+------------------+------------------+------------------+
|Central|Home Office| 91212.64399999991|12438.412399999997|28.398201826484012|
|   West|Home Office|136721.77699999965|16530.414999999957|28.949938704027947|
|  South|Home Office|        74255.0015|4620.6343000000015| 16.98762610294118|
|   East|Home Office| 127463.7259999999| 26709.21680000001|53.205611155378506|
|  South|   Consumer|195580.97100000046|26913.572799999987| 32.11643532219569|
|  South|  Corporate| 121885.9325000001|        15215.2232|29.833770980392156|
+-------+-----------+------------------+------------------+------------------+
only showing top 6 rows


In [80]:
data1.groupby("Region","Segment").agg(sum("Sales").alias("TotalSales"),\
         sum("Profit").alias("TotalProfit"),\
         avg("Profit").alias("AvgProfit")).show(6)

+-------+-----------+------------------+------------------+------------------+
| Region|    Segment|        TotalSales|       TotalProfit|         AvgProfit|
+-------+-----------+------------------+------------------+------------------+
|Central|Home Office| 91212.64399999991|12438.412399999997|28.398201826484012|
|   West|Home Office|136721.77699999965|16530.414999999957|28.949938704027947|
|  South|Home Office|        74255.0015|4620.6343000000015| 16.98762610294118|
|   East|Home Office| 127463.7259999999| 26709.21680000001|53.205611155378506|
|  South|   Consumer|195580.97100000046|26913.572799999987| 32.11643532219569|
|  South|  Corporate| 121885.9325000001|        15215.2232|29.833770980392156|
+-------+-----------+------------------+------------------+------------------+
only showing top 6 rows


In [81]:
data1.groupby("Region").agg(sum("Sales").alias("TotalSales")).show(6) # without sort
data1.groupby("Region").agg(sum("Sales").alias("TotalSales")).sort(desc("TotalSales")).show(6)

+-------+------------------+
| Region|        TotalSales|
+-------+------------------+
|  South|391721.90500000044|
|Central|501239.89080000174|
|   East| 678781.2399999928|
|   West|  725457.824499995|
+-------+------------------+

+-------+------------------+
| Region|        TotalSales|
+-------+------------------+
|   West|  725457.824499995|
|   East| 678781.2399999928|
|Central|501239.89080000174|
|  South|391721.90500000044|
+-------+------------------+



In [82]:
# i want total sales, total profit ans average quantites for each segment and region. which avg qty is greather than 3.5

data1.groupby("Segment","Region")\
    .agg(sum("Sales").alias("TotalSales"),\
         sum("Profit").alias("TotalProfit"),\
         avg("Quantity").alias("AvgQuantity"))\
    .filter(col("AvgQuantity")>3.5) \
    .sort(asc("AvgQuantity")) \
    .show(6)


+-----------+-------+------------------+------------------+------------------+
|    Segment| Region|        TotalSales|       TotalProfit|       AvgQuantity|
+-----------+-------+------------------+------------------+------------------+
|   Consumer|   East|350908.16700000095| 41190.98429999989|3.6398910823689583|
|   Consumer|Central|252031.43400000004| 8564.048100000016|3.7285478547854787|
|Home Office|  South|        74255.0015|4620.6343000000015|3.7316176470588234|
|Home Office|   West|136721.77699999965|16530.414999999957|3.7810858143607704|
|  Corporate|   West|225855.27449999994| 34437.42989999997|           3.78125|
|Home Office|Central| 91212.64399999991|12438.412399999997|3.7831050228310503|
+-----------+-------+------------------+------------------+------------------+
only showing top 6 rows


In [68]:
data1.groupby("Segment","Region")\
    .agg(sum("Sales").alias("TotalSales"),\
         sum("Profit").alias("TotalProfit"),\
         avg("Quantity").alias("AvgQuantity"))\
    .filter((col("AvgQuantity")>3.5) & (col("Region")=="West")) \
    .sort(asc("AvgQuantity")) \
    .show(6)
# Double filter

+-----------+------+------------------+------------------+------------------+
|    Segment|Region|        TotalSales|       TotalProfit|       AvgQuantity|
+-----------+------+------------------+------------------+------------------+
|Home Office|  West|136721.77699999965|16530.414999999957|3.7810858143607704|
|  Corporate|  West|225855.27449999994| 34437.42989999997|           3.78125|
|   Consumer|  West|362880.77300000144|  57450.6040000001| 3.873803827751196|
+-----------+------+------------------+------------------+------------------+



In [ ]:
# i need sub Category, Category total sales, total profit avg quantity

In [99]:
data.groupby("Category","Sub-Category")\
    .agg(sum("Sales"), \
         sum("Profit"),
         avg("Quantity").alias("Avg_Quantity")) \
    .filter((col("Category") == "Technology") & (col("Avg_Quantity") > 3.5 )) \
    .show()

+----------+------------+------------------+-----------------+------------------+
|  Category|Sub-Category|        sum(Sales)|      sum(Profit)|      Avg_Quantity|
+----------+------------+------------------+-----------------+------------------+
|Technology| Accessories|167380.31800000006|41936.63569999997|              3.84|
|Technology|      Phones| 330007.0539999992|44515.73059999997| 3.699662542182227|
|Technology|    Machines|189238.63100000005|        3384.7569|3.8260869565217392|
+----------+------------+------------------+-----------------+------------------+



In [102]:
data1.createOrReplaceTempView("superstoredata")

In [103]:
spark.sql("""
    SELECT 
        Category, 
        SUM(Sales) AS Total_Sales, 
        SUM(Profit) AS Total_Profit, 
        AVG(Quantity) AS Avg_Quantity
    FROM superstoredata
    WHERE Category = 'Technology'
    GROUP BY Category
    ORDER BY SUM(Sales) DESC
""").show()

+----------+-----------------+------------------+-----------------+
|  Category|      Total_Sales|      Total_Profit|     Avg_Quantity|
+----------+-----------------+------------------+-----------------+
|Technology|836154.0329999941|145454.94810000007|3.756903086085544|
+----------+-----------------+------------------+-----------------+

